# DoubleIntegrator — Stage 3 — Unconstrained Online Adaptation

Run online reference adaptation under perturbed dynamics.


## 1. Set up the system and load the stage config

Load dependencies and configuration.


In [ ]:
"""Boilerplate: make the in-repo `sdpc` package importable and resolve this system."""
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_p = Path.cwd()
while not (_p / "src" / "sdpc").exists():
    _p = _p.parent
sys.path.insert(0, str(_p / "src"))

import torch
from sdpc.config import load_config
from sdpc.registry import make_system

SYSTEM = "double_integrator"
device = torch.device("cpu")
system = make_system(SYSTEM, device=device)
CONFIGS = Path.cwd().parent / "configs"
RESULTS = Path.cwd().parent / "results"

print(f"System        : {SYSTEM}")
print(f"State dim nx  : {system.nx}")
print(f"Control dim nu: {system.nu}")
print(f"Sample time ts: {system.ts}")
print(f"Input bounds  : [{system.umin}, {system.umax}]")
print(f"State bounds  : [{system.xmin}, {system.xmax}]")
print(f"Discrete model: {system.is_discrete}")


In [ ]:
cfg = load_config(CONFIGS / 'unconstrained.yaml')
print('Resolved unconstrained-adaptation config:')
for k, v in cfg.items():
    if not k.startswith('_'):
        print(f'  {k}: {v}')

## 2. Load the trained sparse policy

Restore the required checkpoints.


In [ ]:
from sdpc.sindy import load_model

from sdpc.io import find_policy_checkpoint
policy_path = find_policy_checkpoint(RESULTS, cfg)
print('Using policy checkpoint:', policy_path)

policy = load_model(policy_path, device=device)
policy.pretty_print()

## 3. Build the perturbed (deployment) plant

Create the deployment model.


In [ ]:
plant = system.perturbed_plant(cfg)
print('Nominal params  :', system.nominal_params())
print('Perturbed params:', system.perturbed_params(cfg))

## 4. Sample a test scenario (initial state + reference)

Configure the reference signal.


In [ ]:
from sdpc.eval import sample_scenario

data = sample_scenario(system, cfg, cfg.get('seed', 0), device)
print('Initial state:', data['xn'][0, 0].tolist())
print('Reference (first / last):', data['r'][0, 0].tolist(), '/', data['r'][0, -1].tolist())
print('Horizon length:', data['r'].shape[1])

## 5. Baseline: no adaptation under the perturbed plant

Create the deployment model.


In [ ]:
import copy
from sdpc.eval import rollout_closed_loop

res_frozen = rollout_closed_loop(copy.deepcopy(policy), plant, data,
                                 umin=system.umin, umax=system.umax,
                                 action_scale=cfg.get('action_scale', 1.0))
final_err = (data['r'][0, -1] - res_frozen['x_traj'][0, -1]).norm().item()
print(f'Final tracking error with NO adaptation: {final_err:.4f}')

## 6. Run unconstrained adaptation — autograd backend

Compute derivatives with autograd.


In [ ]:
from sdpc.adaptation import run_unconstrained_adaptation, UnconstrainedAdaptationConfig

ucfg_ag = UnconstrainedAdaptationConfig(
    **{**cfg.get('unconstrained', {}), 'ref_backend': 'autograd'}
)
res_ag = run_unconstrained_adaptation(
    copy.deepcopy(policy), plant, data, ucfg_ag,
    umin=system.umin, umax=system.umax,
)
print(f"[autograd]  final track_err = {res_ag['logs'][-1]['track_err']:.4f}")
print(f"[autograd]  mean step time  = "
      f"{sum(l['step_time'] for l in res_ag['logs']) / len(res_ag['logs']):.3e} s")

## 7. Run unconstrained adaptation — symbolic Jacobian backend

Build and verify the sensitivities.


In [ ]:
ucfg_sym = UnconstrainedAdaptationConfig(
    **{**cfg.get('unconstrained', {}), 'ref_backend': 'symbolic'}
)
res_sym = run_unconstrained_adaptation(
    copy.deepcopy(policy), plant, data, ucfg_sym,
    umin=system.umin, umax=system.umax,
)
print(f"[symbolic]  final track_err = {res_sym['logs'][-1]['track_err']:.4f}")
print(f"[symbolic]  mean step time  = "
      f"{sum(l['step_time'] for l in res_sym['logs']) / len(res_sym['logs']):.3e} s")

traj_diff = (res_ag['x_traj'] - res_sym['x_traj']).abs().max().item()
print(f'\nMax trajectory difference between backends: {traj_diff:.2e} '
      '(should be ~0 — both compute the same update)')

## 8. Compare: no adaptation vs. adapted trajectories

Compare frozen and adapted responses.


In [ ]:
from sdpc.plotting import plot_states_and_controls
import matplotlib.pyplot as plt

plot_states_and_controls(
    [
        {'x': res_frozen['x_traj'], 'u': res_frozen['u_traj'], 'label': 'no adaptation', 'color': 'gray'},
        {'x': res_ag['x_traj'],     'u': res_ag['u_traj'],     'label': 'adapted (autograd)', 'color': 'royalblue'},
    ],
    r_traj=data['r'], xmin=system.xmin, xmax=system.xmax,
    title=f'{SYSTEM}: unconstrained adaptation vs. frozen policy',
)
plt.show()

## 9. Tracking error over time

Inspect tracking error over time.


In [ ]:
import matplotlib.pyplot as plt

errs = [l['track_err'] for l in res_ag['logs']]
plt.figure(figsize=(8, 3))
plt.plot(errs)
plt.xlabel('adaptation step'); plt.ylabel('||r - x||')
plt.title(f'{SYSTEM}: tracking error during unconstrained adaptation')
plt.grid(alpha=.3); plt.show()

**Note.** This law has no safety mechanism — nothing here prevents the predicted trajectory from violating state or obstacle constraints while it corrects the tracking offset.
